## Setting up environment for Colab

In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import sys
from google.colab import userdata

config_path = userdata.get('CONFIG_PATH')

sys.path.append(config_path)

from config import BASE_PATH, DATA_PATH, SRC_PATH, SAVED_CHECKPOINTS_PATH

# Week 3 — Session 1 b: Transformers


## Libraries You’ll Use

For this session you’ll need:

- **scikit-learn** — decision tree, random forest, metrics, train/test split  
- **xgboost** — gradient boosting trees  
- **torch** — simple neural network (PyTorch)  
- **matplotlib** & **seaborn** — to visualize decision boundaries and compare models  
- **time** — to measure training and inference time for computational cost analysis
- **transformers** - to create and train the transformer model

---

✅ Make sure you have `scikit-learn`, `xgboost`, `torch`, `seaborn`, and `matplotlib` installed.

In [3]:
!pip install scikit-learn xgboost torch matplotlib seaborn transformers

In [10]:
!pip install {SRC_PATH}

Processing ./drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/src
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.4 MB/s eta 0:00:00
  Created wheel for misinformation_detection: filename=misinformation_detection-1.0.0-py3-none-any.whl size=20155 sha256=c518b6bf1fdf4e171672087f06ecd4e988dd5b6c7d59da2dc6c7f2106319d6a3
  Stored in directory: /tmp/pip-ephem-wheel-cache-h7u81ys1/wheels/46/67/19/6967c24d7ffe355b16d9bd45b2a6576a3fa548c7b58c981aee
Successfully built misinformation_detection


## Step 1 — Preparing the Data: Combining Structured & Text Features

In this session, we’re building **advanced models** that can learn from both:
- **Structured metadata & linguistic features** (like word count, readability, clickbait flag, source domain one-hots)
- **Unstructured text features** (using TF-IDF on our lemmatized article text)

---

### Why Combine Both?

- Metadata and linguistic cues often provide powerful signals that text alone might miss.
- TF-IDF captures word-level patterns, while numerical features add context about source, style, and structure.

---

### How We’ll Do It

If you recall, we already created the function that allows us to combine TFIDF features with the other structured features we created earlier. The steps followed were:

1. **Split the data** into train and test sets first (to avoid data leakage).  
2. **Fit the TF-IDF vectorizer only on the training text** and transform the test text with the same vocabulary.  
3. **Combine** the sparse TF-IDF matrix with the numerical features to create a single feature matrix for each split.

---

**By merging these together**, we give our decision tree and random forest models a **richer, more diverse set of signals** — boosting their ability to detect subtle patterns in misinformation.

In [11]:
import sys
import os
from misinformation_detection.data import prepare_text_structured_features_full
from pathlib import Path

X_train_combined, X_test_combined, X_structured_train_full, X_structured_test_full, y_train, y_test, vectorizer = prepare_text_structured_features_full(
    DATA_PATH / Path("processed/fnn_lemmatized.csv")
)

## Using Transformers for Misinformation Detection

In this part, we’ll introduce a **transformer-based model** to tackle the misinformation detection problem using only text.

---

### Why Transformers?

- Transformers like are state-of-the-art for **natural language understanding**.
- They capture complex **contextual relationships** in text that classical models or simple TF-IDF cannot.
- Fine-tuning a pre-trained transformer on your dataset can yield **richer text representations**, which is valuable for nuanced tasks like misinformation detection.

---

**Key takeaway:**  
Transformers give you a **strong text-only baseline**, helping you see the value of modern NLP models for misinformation detection and how they differ from classical approaches.

## Building a Frozen DistilBert Encoder with MLP Classifier

In this step, we define a custom PyTorch model `DistilBert` that uses a **frozen DistilBert encoder** and an additional **multi-layer perceptron (MLP)** classifier head for binary classification.

### Dataset Preparation and Tokenization

We begin by preparing the data for training with transformers:

1. **Label Encoding**: Convert `fake` and `real` into binary labels (`0` and `1`).
2. **Train/Validation/Test Split**:
   - 80% training
   - 10% validation
   - 10% test
   - Stratified to preserve class balance.
3. **Tokenization**:
   - We use the `DistilBert tokenizer` to tokenize the `article_cleaned` text.
   - Input is truncated or padded to a maximum length of 512 tokens.
4. **PyTorch Dataset Wrapping**:
   - We define a custom `FNNDataset` to wrap the tokenized inputs and labels.
   - These datasets will be passed into a `DataLoader` during training.

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.optim import AdamW
from tqdm import tqdm
import pandas as pd

# Load the cleaned dataset
fnn_df = pd.read_csv(DATA_PATH / Path("processed/fnn_cleaned.csv"))
print(f"Loaded shape: {fnn_df.shape}")
print(fnn_df.columns.tolist())

# 1. Encode labels
fnn_df["label_enc"] = fnn_df["label"].map({"fake": 0, "real": 1})

# 2. Train/Val/Test split
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    fnn_df["article_cleaned"].tolist(),
    fnn_df["label_enc"].tolist(),
    test_size=0.2,
    stratify=fnn_df["label_enc"],
    random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)
print(f"Train size: {len(train_texts)}, Val size: {len(val_texts)}, Test size: {len(test_texts)}")

# 3. Tokenizer & encodings
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
def encode(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

train_enc = encode(train_texts)
val_enc   = encode(val_texts)
test_enc  = encode(test_texts)
print(f"Train encodings shape: {train_enc['input_ids'].shape}")

# 4. Dataset wrapper
class FNNDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

train_ds = FNNDataset(train_enc, train_labels)
val_ds   = FNNDataset(val_enc,   val_labels)
test_ds  = FNNDataset(test_enc,  test_labels)

# 5. DataLoaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)

Loaded shape: (13892, 25)
['id', 'title', 'label', 'dataset_source', 'source_domain', 'article_cleaned', 'date_cleaned', 'source_domain.1', 'source_domain_grouped', 'source_Other', 'source_billboard.com', 'source_dailymail.co.uk', 'source_elle.com', 'source_en.wikipedia.org', 'source_etonline.com', 'source_ew.com', 'source_harpersbazaar.com', 'source_hollywoodreporter.com', 'source_inquisitr.com', 'source_people.com', 'source_radaronline.com', 'source_thewrap.com', 'source_today.com', 'source_usmagazine.com', 'source_variety.com']
Train size: 11113, Val size: 1389, Test size: 1390


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train encodings shape: torch.Size([11113, 512])


### Load DistilBERT Model and Setup Device

We now load the **distilbert-base-uncased** transformer model for binary classification (`fake` vs. `real`). This approach leverages the `[CLS]` token embedding from the transformer as a compact representation of the input text.

- The **DistilBert encoder** is loaded from the pretrained `"DistilBert"` model and its parameters are **frozen** (i.e., not updated during training).
- The classifier is a 2-layer MLP:
  - A dropout layer to prevent overfitting.
  - A hidden linear layer followed by ReLU activation.
  - A second dropout layer.
  - A final linear layer that outputs logits for 2 classes (real vs. fake).

We will train only the classifier head while keeping the encoder fixed, allowing for faster training and less risk of overfitting on limited data.

To utilize available hardware acceleration, we check for:
-  **CUDA** (NVIDIA GPUs)
-  **MPS** (Apple Silicon GPUs)
-  Fall back to **CPU** if no accelerator is found.

This ensures the model runs efficiently on different hardware setups.

In [17]:
# 6. Model definition
class DistilBERTClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained("distilbert-base-uncased")
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Linear(self.encoder.config.hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_tok = out.last_hidden_state[:, 0, :]
        return self.classifier(cls_tok)

# 7. Device selection: MPS → CUDA → CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

model = DistilBERTClassifier().to(device)

Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Training with Frozen DistilBert Encoder (Partial Fine-Tuning)

Instead of **fine-tuning all model parameters**, here we **freeze the DistillBERT encoder** and **train only the classifier head**.

- **Loss Function**: We use `CrossEntropyLoss` since we’re manually handling logits instead of model-internal loss.
- **Optimizer**: `AdamW` is applied **only to the classifier parameters** (`model.classifier.parameters()`), which are the only trainable parts.
- **Training**: The forward pass uses the frozen encoder to produce `[CLS]` embeddings, which are passed through the MLP classifier.
- This setup allows us to **train faster** and reduces the risk of overfitting when labeled data is limited.

This configuration is ideal for using pretrained models in low-resource settings without incurring the computational cost of full fine-tuning.

In [14]:
# 8. Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.classifier.parameters(), lr=1e-5)

# 9. Training & evaluation functions
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids, attention_mask)
            preds += torch.argmax(logits, dim=1).cpu().tolist()
            targets += labels.cpu().tolist()
    acc = accuracy_score(targets, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(targets, preds, average="binary")
    print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

def train(model, train_loader, val_loader, epochs=3):
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch} Train Loss: {total_loss/len(train_loader):.4f}")
        evaluate(model, val_loader)

In [ ]:
# 10. Run training & final test evaluation
train(model, train_loader, val_loader, epochs=5)
print("Test set performance:")
evaluate(model, test_loader)

Epoch 1: 100%|██████████| 695/695 [03:05<00:00,  3.75it/s]


Epoch 1 Train Loss: 0.5659
Acc: 0.7696 | Prec: 0.7696 | Rec: 1.0000 | F1: 0.8698


Epoch 2: 100%|██████████| 695/695 [03:03<00:00,  3.78it/s]


Epoch 2 Train Loss: 0.5212
Acc: 0.7696 | Prec: 0.7696 | Rec: 1.0000 | F1: 0.8698


Epoch 3: 100%|██████████| 695/695 [03:04<00:00,  3.77it/s]


Epoch 3 Train Loss: 0.5001
Acc: 0.7696 | Prec: 0.7696 | Rec: 1.0000 | F1: 0.8698


Epoch 4: 100%|██████████| 695/695 [03:03<00:00,  3.79it/s]


Epoch 4 Train Loss: 0.4759
Acc: 0.7847 | Prec: 0.7814 | Rec: 1.0000 | F1: 0.8773


Epoch 5: 100%|██████████| 695/695 [03:03<00:00,  3.78it/s]


Epoch 5 Train Loss: 0.4559
Acc: 0.8006 | Prec: 0.7960 | Rec: 0.9963 | F1: 0.8849
Test set performance:
Acc: 0.8014 | Prec: 0.7974 | Rec: 0.9944 | F1: 0.8851


In [21]:
import torch

model.load_state_dict(torch.load(SAVED_CHECKPOINTS_PATH / Path("frozen_transformer.pt")))

<All keys matched successfully>

In [ ]:
model_save_path = SAVED_CHECKPOINTS_PATH / Path("frozen_transformer.pt")
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to /content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/data/saved_checkpoints/frozen_transformer.pt


In [24]:
import time

# --- Inference Time ---
start_time = time.time()
# Run evaluation on the test set
evaluate(model, test_loader)
end_time = time.time()

# Calculate time per sample (using the size of the final test set)
nn_inference_time = (end_time - start_time) / len(test_ds)
print(f"Neural Network Inference Time: {nn_inference_time:.6f} seconds/sample")

Acc: 0.8014 | Prec: 0.7974 | Rec: 0.9944 | F1: 0.8851
Neural Network Inference Time: 0.014425 seconds/sample


## What if we don't freeze anything? (optional further exploration)

Instead of freezing our encoder, we have the option to train all the parameters in the model. We will see in this section that this process takes much longer and may not be worth it for the resulting performance improvement.

### What We’ll Do

- Load a **pre-trained RoBERTa model** (`roberta-base`).
- Tokenize the raw cleaned article texts (not lemmatized).
- **First**, we’ll **fine-tune the entire model end-to-end** on the classification task.
- **Then**, we’ll freeze the transformer and train a **shallow neural network** on the **[CLS] token** to predict real or fake.
- Finally, we’ll evaluate the transformer’s performance **independently** and compare it to tree-based and ensemble models.

---

## Load Cleaned Data for Transformer

Before we fine-tune our transformer model, we’ll load the **cleaned FakeNewsNet dataset** (`fnn_cleaned.csv`).  
This dataset includes the lemmatized article texts and labels we need for training.


In [25]:
import pandas as pd
from misinformation_detection.data import load_fakenewsnet_from_dataframe

# Load the cleaned dataset
fnn_df = load_fakenewsnet_from_dataframe(DATA_PATH / Path("processed/fnn_cleaned.csv"), verbose=True)

Loaded FakeNewsNet cleaned dataset with 13,892 rows
                     id                                              title  \
0  gossipcop-2493749932  Did Miley Cyrus and Liam Hemsworth secretly ge...   
1   gossipcop-941805037  Celebrities Join Tax March in Protest of Donal...   

  label dataset_source    source_domain  \
0  fake      gossipcop  dailymail.co.uk   
1  fake      gossipcop      variety.com   

                                     article_cleaned date_cleaned  \
0  congratulations might be in order for miley cy...   22/06/2018   
1  thousands are taking the streets to protest pr...   15/04/2017   

   source_domain.1 source_domain_grouped  source_Other  ...  source_ew.com  \
0  dailymail.co.uk       dailymail.co.uk         False  ...          False   
1      variety.com           variety.com         False  ...          False   

   source_harpersbazaar.com  source_hollywoodreporter.com  \
0                     False                         False   
1                 

### Dataset Preparation and Tokenization

We begin by preparing the data for training with transformers:

1. **Label Encoding**: Convert `fake` and `real` into binary labels (`0` and `1`).
2. **Train/Validation/Test Split**:
   - 80% training
   - 10% validation
   - 10% test
   - Stratified to preserve class balance.
3. **Tokenization**:
   - We use the `RoBERTa tokenizer` to tokenize the `article_cleaned` text.
   - Input is truncated or padded to a maximum length of 512 tokens.
4. **PyTorch Dataset Wrapping**:
   - We define a custom `FNNDataset` to wrap the tokenized inputs and labels.
   - These datasets will be passed into a `DataLoader` during training.

In [26]:
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer
import torch

# Encode labels
fnn_df["label_enc"] = fnn_df["label"].map({"fake": 0, "real": 1})

# 80% train, 10% val, 10% test split
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    fnn_df["article_cleaned"].tolist(), fnn_df["label_enc"].tolist(),
    test_size=0.2, stratify=fnn_df["label_enc"], random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

# Tokenization
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Dataset wrapper
class FNNDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx]) for key, val in self.encodings.items()
        } | {"labels": torch.tensor(self.labels[idx])}

# Create datasets
train_dataset = FNNDataset(train_encodings, train_labels)
val_dataset = FNNDataset(val_encodings, val_labels)
test_dataset = FNNDataset(test_encodings, test_labels)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

### Create DataLoaders

We now wrap our dataset splits into PyTorch `DataLoader` objects for efficient batching and shuffling during training.

- **Train Loader**: Shuffled for learning
- **Validation & Test Loaders**: No shuffling (used for evaluation)

Batch size is set to **16**

In [27]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

print("DataLoaders ready: Train, Validation, Test")

DataLoaders ready: Train, Validation, Test


### Load RoBERTa Model and Setup Device

We now load the **RoBERTa-base** transformer model for binary classification (`fake` vs. `real`). The model's `[CLS]` token embedding is used by default for classification.

To utilize available hardware acceleration, we check for:
-  **CUDA** (NVIDIA GPUs)
-  **MPS** (Apple Silicon GPUs)
-  Fall back to **CPU** if no accelerator is found.

This ensures the model runs efficiently on different hardware setups.

In [28]:
from transformers import RobertaForSequenceClassification

# Load RoBERTa model for binary classification
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

# Device selection: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Move model to the selected device
model.to(device)

print(f"Model loaded and moved to device: {device}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to device: cuda


### Training & Evaluation Loop

We now define the training and evaluation logic for our transformer-based classifier. This includes:

- **Full fine-tuning** of the model — we update **all parameters** of the pretrained transformer (no layers are frozen).
- Initializing the **AdamW** optimizer with a learning rate of `2e-5`.
- A `train()` loop that iterates through batches, calculates loss, performs backpropagation, and evaluates after each epoch.
- An `evaluate()` function that computes accuracy, precision, recall, and F1 score on the validation set.

This setup will help us track learning progress and model generalization throughout training.

In [29]:
import torch
from transformers import RobertaForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.optim import AdamW
from tqdm import tqdm

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training loop
def train(model, train_loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Average Training Loss: {avg_loss:.4f}")

        evaluate(model, val_loader)

# Evaluation loop
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, preds, average="binary")
    print(f"Validation - Acc: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

In [ ]:
# Run training
train(model, train_loader, val_loader, epochs=1)

Training Epoch 1: 100%|██████████| 695/695 [18:48<00:00,  1.62s/it]


Epoch 1 Average Training Loss: 0.4280
Validation - Acc: 0.8395, Precision: 0.8685, Recall: 0.9326, F1: 0.8994


In [ ]:
# Final evaluation on test set
evaluate(model, test_loader)

Validation - Acc: 0.8353, Precision: 0.8659, Recall: 0.9298, F1: 0.8967


In [ ]:
# save model
model_save_path = SAVED_CHECKPOINTS_PATH / Path("roberta_fnn_model_2.pt")
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to /content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/data/saved_checkpoints/roberta_fnn_model_2.pt


In [31]:
model.load_state_dict(torch.load(SAVED_CHECKPOINTS_PATH / Path("roberta_fnn_model_2.pt")))

<All keys matched successfully>

In [32]:
import time

# --- Inference Time ---
start_time = time.time()
# Run evaluation on the test set
evaluate(model, test_loader)
end_time = time.time()

# Calculate time per sample (using the size of the final test set)
nn_inference_time = (end_time - start_time) / len(test_ds)
print(f"Neural Network Inference Time: {nn_inference_time:.6f} seconds/sample")

Validation - Acc: 0.8353, Precision: 0.8659, Recall: 0.9298, F1: 0.8967
Neural Network Inference Time: 0.028831 seconds/sample


## Computational Cost Comparison

To evaluate the computational efficiency of different models, we compare their training and inference times as well as resource usage. The table below summarizes these metrics for all models tested.


| Model                  | Training Time (s) | Inference Time (s/sample) | Hardware Used       |
|------------------------|------------------|---------------------------|---------------------|
| Decision Tree          | 1.2437           |  0.000002                 | CPU                 |
| Random Forest          | 6.6199           | 0.000022                  | CPU                 |
| XGBoost                | 10.0593          | 0.000015                  | CPU                 |
| Simple Neural Network  | 0.68323 s(/epoch) | 0.000024                  | CPU             |
| Transformer Fine-Tuned | 183.4583 (s/epoch)  | 0.014425               | GPU                 |
| Transformer Fine-Tuned | 1128.2313 (s/epoch) | 0.028831               | GPU                 |


This comparison provides insight into the trade-offs between model complexity, training duration, and real-time usability for misinformation detection tasks.


## Let's pack everything up!

Same as we've been doing in each session, it's time we neatly package these models into classes and save them in our src folder!

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

class FNNTransformerDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

class DistilBERTClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        for param in self.encoder.parameters():
            param.requires_grad = False  # freeze encoder

        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = encoder_outputs.last_hidden_state[:, 0, :]
        cls_token = self.dropout(cls_token)
        logits = self.classifier(cls_token)
        return logits

class DistilBERTModelWrapper:
    def __init__(self, device=None):
        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        self.model = DistilBERTClassifier().to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.classifier.parameters(), lr=1e-5)

    def encode_texts(self, texts, max_length=512):
        return self.tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors='pt')

    def create_dataloader(self, texts, labels, batch_size=16, shuffle=True):
        encodings = self.encode_texts(texts)
        dataset = FNNTransformerDataset(encodings, labels)
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    def train(self, train_loader, val_loader, epochs=3):
        best_val_loss = float('inf')
        best_model_state = None

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(input_ids, attention_mask)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            avg_train_loss = total_loss / len(train_loader)
            val_loss, val_metrics = self.evaluate(val_loader)
            print(f"Epoch {epoch+1}: Train loss = {avg_train_loss:.4f}, Val loss = {val_loss:.4f}, Val acc = {val_metrics['accuracy']:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = self.model.state_dict()

        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)

    def evaluate(self, dataloader):
        self.model.eval()
        total_loss = 0
        preds, targets = [], []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                outputs = self.model(input_ids, attention_mask)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()

                predictions = torch.argmax(outputs, dim=1)
                preds.extend(predictions.cpu().numpy())
                targets.extend(labels.cpu().numpy())

        avg_loss = total_loss / len(dataloader)
        accuracy = accuracy_score(targets, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(targets, preds, average='binary')
        metrics = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}
        print(f"Eval — Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
        return avg_loss, metrics

    def save(self, filepath):
        torch.save(self.model.state_dict(), filepath)
        print(f"Model saved to {filepath}")

    def load(self, filepath):
        self.model.load_state_dict(torch.load(filepath, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()
        print(f"Model loaded from {filepath}")
